In [ ]:
% pip install scikit-learn
% pip install matplotlib
% pip install numpy

Replace the path's with the correct paths to your results generated using each prompting method with the respective models. Wherever the path is mentioned `with open`, please replace with your path to the respective file.

```python
with open(".\llama3.211BV\llama3.2_11B_zshot_predictions.json") as file:
```

In [17]:
def transform_label_dict(label_dict):
    # Define the mapping between class labels and the corresponding descriptions
    label_mapping = {
        'class_label_1': 'Missing fixation',
        'class_label_2': 'Reduced fixation duration',
        'class_label_3': 'Undefined reason'
    }

    transformed_dict = {}

    for key, value in label_dict.items():
        transformed_dict[key] = {}

        # Map each label to its new description based on `label_mapping`
        for class_label, new_description in label_mapping.items():
            transformed_dict[key][new_description] = value.get(class_label, 0)
        
        # Set "No Missing Subgraph" based on all class labels being 0
        transformed_dict[key]["No Missing Subgraph"] = int(all(
            value.get(class_label, 0) == 0 for class_label in label_mapping
        ))

    return transformed_dict

In [20]:
from sklearn.metrics import classification_report, accuracy_score, hamming_loss, roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
from sklearn.metrics import precision_recall_curve, roc_curve, precision_recall_fscore_support
import numpy as np
import matplotlib.pyplot as plt


"""
https://scikit-learn.org/1.5/modules/generated/sklearn.metrics.classification_report.html
https://scikit-learn.org/1.5/modules/generated/sklearn.metrics.accuracy_score.html
https://scikit-learn.org/1.5/modules/generated/sklearn.metrics.roc_auc_score.html#roc-auc-score
"""

def evaluation_metrics(predictions: list[dict], ground_truth: list[dict]):
    # Predictions and ground truth data
    labels = set({
    "Missing fixation",
    "Reduced fixation duration",
    "Undefined reason",
    "No Missing Subgraph"
    })
    
    y_true = np.array([[gt[label] for label in labels] for gt in ground_truth])
    y_pred = np.array([[pred[label] for label in labels] for pred in predictions])

    # Calculate multilabel classification metrics
    print("Classification Report:")
    print(classification_report(y_true, y_pred, target_names=labels, digits=4))

    print("\nAccuracy Score:", accuracy_score(y_true, y_pred))
    print("Hamming Loss:", hamming_loss(y_true, y_pred))
    print("ROC AUC Score:", roc_auc_score(y_true, y_pred, average='macro', multi_class='ovr'))
    print()

    # Calculate ROC AUC and Precision-Recall AUC for each label
    for i, label in enumerate(labels):
        try:
            roc_auc = roc_auc_score(y_true[:, i], y_pred[:, i])
            avg_precision = average_precision_score(y_true[:, i], y_pred[:, i])
            accuracy = accuracy_score(y_true[:, i], y_pred[:, i])

            print(f"Accuracy for {label}: {accuracy}")
            print(f"ROC AUC for {label}: {roc_auc}")
            print("Hamming loss", hamming_loss(y_true[:, i], y_pred[:, i]))
            print("--------------------------------------------------")
        
        except ValueError:
            print(f"\nROC AUC and Average Precision for {label} could not be calculated due to lack of positive samples.")

In [22]:
import json

# Replace with the actual file paths

metadata_file = "../original_fixation_transcript_label_missed.json"
data_file = "../original_fixation_transcript_simulated_error_data.json"

with open(metadata_file, 'r') as file:
    orig_xy_ground_truth_metadata = json.load(file)

with open(data_file, 'r') as file:
    orig_xy_fixation_data = json.load(file)

# GPT-4o-Mini

## MAARTA

In [23]:
import json

# Replace with the actual file path
result_file = "../missed_findings_results.json"

with open (result_file) as file:
    gpt4oMini_saved_results = json.load(file)

predictions = []
transformed_gt = transform_label_dict(orig_xy_ground_truth_metadata)
ground_truth = []

for dicom_id, pred in gpt4oMini_saved_results.items():
    predictions.append({
            "Missing fixation": pred["Missing fixation"],
            "Reduced fixation duration": pred["Reduced fixation duration"],
            "Undefined reason": pred["Undefined reason"],
            "No Missing Subgraph": pred["No Missing Subgraph"],
        })
        
    ground_truth.append(transformed_gt[dicom_id])

assert len(predictions) == len(ground_truth)

evaluation_metrics(predictions, ground_truth)

Classification Report:
                           precision    recall  f1-score   support

Reduced fixation duration     0.8429    1.0000    0.9147        59
         Missing fixation     0.6092    0.9815    0.7518        54
         Undefined reason     0.9444    0.5312    0.6800        32
      No Missing Subgraph     1.0000    1.0000    1.0000        50

                micro avg     0.7956    0.9179    0.8524       195
                macro avg     0.8491    0.8782    0.8366       195
             weighted avg     0.8351    0.9179    0.8529       195
              samples avg     0.8294    0.9077    0.8542       195


Accuracy Score: 0.7440476190476191
Hamming Loss: 0.09226190476190477
ROC AUC Score: 0.8882769368846578

Accuracy for Reduced fixation duration: 0.9345238095238095
ROC AUC for Reduced fixation duration: 0.9495412844036697
Hamming loss 0.06547619047619048
--------------------------------------------------
Accuracy for Missing fixation: 0.7916666666666666
ROC AUC for Mis